# Compare author-annotated CTs with HRApop CTann annotations

This notebook benchmarks the cell types (CTs) assigned by HRApop CT annotation tools against the author-provided CT for the **same `(dataset, cell)`**.

### Main choices
- **Reference:** rows where `tool == "author"`
- **Prediction:** CellTypist, Azimuth, Pan-Human Azimuth, popV, and FR-Match rows
- **Primary correctness criterion:** exact equality of Cell Ontology IDs (`cell_id`), not label text
- **Coverage and accuracy are reported separately**
- Results are summarized overall, by organ, by dataset, by confidence bin, and by common cells covered by all major tools

> Note: exact CL-ID agreement is intentionally strict. A prediction that is a biologically valid parent/child term is still counted as a mismatch here. An ontology-aware extension can classify these later as exact, broader, narrower, related, or unrelated.

In [ ]:
from pathlib import Path
import csv
import gzip
import collections
import math

# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
# Change this path if the input file is somewhere else on your computer.
INPUT_FILE = Path("anatomogram-sc-transcriptomics-cell-instances.csv.gz")

# In ChatGPT's runtime the uploaded file is mounted here, so use it automatically
# if a local relative file is not present.
if not INPUT_FILE.exists():
    chatgpt_path = Path("/mnt/data/anatomogram-sc-transcriptomics-cell-instances.csv.gz")
    if chatgpt_path.exists():
        INPUT_FILE = chatgpt_path

OUTPUT_DIR = Path("ctann_vs_author_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ORG_NAMES = {
    "UBERON:0002107": "Liver",
    "UBERON:0001264": "Pancreas",
    "UBERON:0002113": "Kidney",
    "UBERON:0002048": "Lung",
}

TOOLS = ["celltypist", "azimuth", "pan-human-azimuth", "popv", "frmatch"]
MAJOR_TOOLS = ["celltypist", "azimuth", "pan-human-azimuth", "popv"]

print("Input:", INPUT_FILE.resolve())
print("Output directory:", OUTPUT_DIR.resolve())

## 1. Load and reorganize the HRApop cell-instance table

The input is in long format: the same biological cell can have one `author` row and one row for each CTann tool. We reorganize it as:

```text
(dataset, cell) -> {tool_name: row}
```

Including `dataset` in the key prevents identical cell barcodes from different datasets from being accidentally merged.

If duplicate `(dataset, cell, tool)` rows occur, the code keeps the row with the higher confidence score and counts the duplicate.

In [ ]:
def as_float(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return float("nan")


rec = collections.defaultdict(dict)
dups = collections.Counter()
rows_by_tool = collections.Counter()

with gzip.open(INPUT_FILE, "rt", newline="", encoding="utf-8-sig") as f:
    reader = csv.DictReader(f)
    required = {"dataset", "organ", "tool", "cell", "cell_id", "cell_label", "match_type", "confidence_score"}
    missing = required - set(reader.fieldnames or [])
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    for row in reader:
        key = (row["dataset"], row["cell"])
        tool = row["tool"]
        rows_by_tool[tool] += 1

        if tool in rec[key]:
            dups[tool] += 1
            new_conf = as_float(row["confidence_score"])
            old_conf = as_float(rec[key][tool]["confidence_score"])

            if (not math.isnan(new_conf)) and (math.isnan(old_conf) or new_conf > old_conf):
                rec[key][tool] = row
        else:
            rec[key][tool] = row

print(f"Unique (dataset, cell) keys: {len(rec):,}")
print("Rows by tool:", dict(rows_by_tool))
print("Duplicate dataset/cell/tool rows:", dict(dups))

## 2. Metric function

For each tool we compare the author's `cell_id` with the tool's predicted `cell_id`.

- **Exact agreement** = fraction of overlapping cells with identical CL IDs
- **Macro precision / recall / F1** = unweighted average across CT classes present in author annotations
- **Weighted F1** = per-class F1 weighted by the number of author cells in each CT
- Confidence means are also reported separately for exact matches and mismatches.

In [ ]:
def metrics_for(pairs):
    """Compute strict CL-ID classification metrics.

    Pair tuple layout:
      (author_id, predicted_id, confidence,
       author_label, predicted_label, match_type,
       organ_id, dataset, cell)
    """
    n = len(pairs)
    exact = sum(author_id == pred_id for author_id, pred_id, *_ in pairs)

    actual = collections.Counter(author_id for author_id, pred_id, *_ in pairs)
    predicted = collections.Counter(pred_id for author_id, pred_id, *_ in pairs)
    true_positive = collections.Counter(
        author_id
        for author_id, pred_id, *_ in pairs
        if author_id == pred_id
    )

    # Average over classes present in the author annotations.
    classes = sorted(actual)
    precisions = []
    recalls = []
    f1s = []
    weighted_f1_sum = 0.0

    for cell_type in classes:
        tp = true_positive[cell_type]
        precision = tp / predicted[cell_type] if predicted[cell_type] else 0.0
        recall = tp / actual[cell_type] if actual[cell_type] else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

        precisions.append(precision)
        recalls.append(recall)
        f1s.append(f1)
        weighted_f1_sum += f1 * actual[cell_type]

    conf_match = [
        as_float(x[2]) for x in pairs
        if x[0] == x[1] and not math.isnan(as_float(x[2]))
    ]
    conf_mismatch = [
        as_float(x[2]) for x in pairs
        if x[0] != x[1] and not math.isnan(as_float(x[2]))
    ]

    return {
        "overlap_cells": n,
        "exact_matches": exact,
        "mismatches": n - exact,
        "exact_agreement": exact / n if n else float("nan"),
        "macro_precision": sum(precisions) / len(precisions) if precisions else float("nan"),
        "macro_recall": sum(recalls) / len(recalls) if recalls else float("nan"),
        "macro_f1": sum(f1s) / len(f1s) if f1s else float("nan"),
        "weighted_f1": weighted_f1_sum / n if n else float("nan"),
        "author_classes": len(actual),
        "predicted_classes": len(predicted),
        "mean_conf_match": sum(conf_match) / len(conf_match) if conf_match else float("nan"),
        "mean_conf_mismatch": sum(conf_mismatch) / len(conf_mismatch) if conf_mismatch else float("nan"),
    }

## 3. Build author-vs-tool cell pairs

Only cells with an `author` annotation can be benchmarked. Missing predictions are **not** counted as errors; instead they affect tool coverage.

In [ ]:
pairs_by_tool = collections.defaultdict(list)
pairs_by_tool_org = collections.defaultdict(list)
pairs_by_tool_dataset = collections.defaultdict(list)
mismatch_rows = []

all_author_cells = 0
all_author_with_any_prediction = 0
any_tool_exact_match = 0

for key, tool_map in rec.items():
    if "author" not in tool_map:
        continue

    all_author_cells += 1
    author = tool_map["author"]
    author_id = author["cell_id"]
    predictions = []

    for tool in TOOLS:
        if tool not in tool_map:
            continue

        pred = tool_map[tool]
        conf = as_float(pred["confidence_score"])

        item = (
            author_id,
            pred["cell_id"],
            conf,
            author["cell_label"],
            pred["cell_label"],
            pred["match_type"],
            author["organ"],
            author["dataset"],
            author["cell"],
        )

        pairs_by_tool[tool].append(item)
        pairs_by_tool_org[(tool, author["organ"])].append(item)
        pairs_by_tool_dataset[(tool, author["dataset"])].append(item)
        predictions.append((tool, pred["cell_id"], conf))

        if author_id != pred["cell_id"]:
            mismatch_rows.append({
                "dataset": author["dataset"],
                "organ_id": author["organ"],
                "organ": ORG_NAMES.get(author["organ"], author["organ"]),
                "cell": author["cell"],
                "tool": tool,
                "author_cell_id": author_id,
                "author_cell_label": author["cell_label"],
                "pred_cell_id": pred["cell_id"],
                "pred_cell_label": pred["cell_label"],
                "pred_match_type": pred["match_type"],
                "confidence_score": pred["confidence_score"],
            })

    if predictions:
        all_author_with_any_prediction += 1
        if any(pred_id == author_id for _, pred_id, _ in predictions):
            any_tool_exact_match += 1

print(f"Author cells: {all_author_cells:,}")
print(f"Author cells with >=1 CTann prediction: {all_author_with_any_prediction:,}")
print(f"Author cells with >=1 exact CTann match: {any_tool_exact_match:,}")

## 4. Overall per-tool summary

`author_coverage` answers **how many author-annotated cells the tool can be compared on**. `exact_agreement` answers **among those overlapping cells, how often the CL ID is identical**.

In [ ]:
summary = []

for tool in TOOLS:
    metrics = metrics_for(pairs_by_tool[tool])
    covered = metrics["overlap_cells"]

    summary.append({
        "tool": tool,
        "tool_rows": rows_by_tool[tool],
        "duplicate_cell_tool_rows": dups[tool],
        "author_cells_total": all_author_cells,
        "author_cells_with_tool": covered,
        "author_coverage": covered / all_author_cells if all_author_cells else float("nan"),
        **metrics,
    })

# Pretty display if pandas is available; otherwise print dictionaries.
try:
    import pandas as pd
    summary_df = pd.DataFrame(summary)
    display(summary_df)
except ImportError:
    summary_df = None
    for row in summary:
        print(row)

## 5. Organ-level summary

In [ ]:
organ_summary = []

for tool in TOOLS:
    for organ_id in sorted(ORG_NAMES):
        pairs = pairs_by_tool_org[(tool, organ_id)]
        if not pairs:
            continue

        organ_summary.append({
            "tool": tool,
            "organ_id": organ_id,
            "organ": ORG_NAMES[organ_id],
            **metrics_for(pairs),
        })

try:
    organ_df = pd.DataFrame(organ_summary)
    display(organ_df)

    # Convenient organ x tool exact-agreement matrix.
    display(
        organ_df.pivot(index="organ", columns="tool", values="exact_agreement")
        .sort_index()
    )
except (NameError, ImportError):
    for row in organ_summary:
        print(row)

## 6. Dataset-level summary

In [ ]:
dataset_summary = []

for (tool, dataset), pairs in sorted(pairs_by_tool_dataset.items()):
    organ_id = pairs[0][6] if pairs else ""

    dataset_summary.append({
        "tool": tool,
        "dataset": dataset,
        "organ_id": organ_id,
        "organ": ORG_NAMES.get(organ_id, organ_id),
        **metrics_for(pairs),
    })

try:
    dataset_df = pd.DataFrame(dataset_summary)
    display(dataset_df)
except (NameError, ImportError):
    for row in dataset_summary[:20]:
        print(row)

## 7. Top author → CTann confusion pairs

This counts the most frequent strict CL-ID disagreements for each tool. These pairs are useful for spotting ontology granularity issues such as a broad author term versus a more specific CTann term.

In [ ]:
confusion_counts = collections.Counter()
label_lookup = {}

for tool, pairs in pairs_by_tool.items():
    for author_id, pred_id, conf, author_label, pred_label, *_ in pairs:
        if author_id != pred_id:
            confusion_counts[(tool, author_id, pred_id)] += 1
            label_lookup[(tool, author_id, pred_id)] = (author_label, pred_label)

confusion_rows = []

for tool in TOOLS:
    tool_confusions = [
        (count, author_id, pred_id)
        for (this_tool, author_id, pred_id), count in confusion_counts.items()
        if this_tool == tool
    ]
    tool_confusions.sort(reverse=True)

    for rank, (count, author_id, pred_id) in enumerate(tool_confusions[:25], 1):
        author_label, pred_label = label_lookup[(tool, author_id, pred_id)]
        confusion_rows.append({
            "tool": tool,
            "rank": rank,
            "count": count,
            "author_cell_id": author_id,
            "author_cell_label": author_label,
            "pred_cell_id": pred_id,
            "pred_cell_label": pred_label,
        })

try:
    confusion_df = pd.DataFrame(confusion_rows)
    display(confusion_df)
except (NameError, ImportError):
    for row in confusion_rows:
        print(row)

## 8. Confidence-bin analysis

This asks whether higher-confidence CTann calls are more likely to exactly match the author CL ID.

In [ ]:
CONFIDENCE_BINS = [
    (0.0, 0.5, "[0,0.5)"),
    (0.5, 0.7, "[0.5,0.7)"),
    (0.7, 0.8, "[0.7,0.8)"),
    (0.8, 0.9, "[0.8,0.9)"),
    (0.9, 0.95, "[0.9,0.95)"),
    (0.95, 1.0000001, "[0.95,1]"),
]

confidence_summary = []

for tool in TOOLS:
    pairs = pairs_by_tool[tool]

    for low, high, label in CONFIDENCE_BINS:
        subset = [
            x for x in pairs
            if not math.isnan(x[2]) and low <= x[2] < high
        ]

        if not subset:
            continue

        confidence_summary.append({
            "tool": tool,
            "confidence_bin": label,
            **metrics_for(subset),
        })

try:
    confidence_df = pd.DataFrame(confidence_summary)
    display(confidence_df)
except (NameError, ImportError):
    for row in confidence_summary:
        print(row)

## 9. Fair head-to-head comparison on common cells

Overall tool metrics can involve different cell populations because tool coverage differs. This section restricts the benchmark to cells having **author + all four major CTann outputs** so every tool is evaluated on exactly the same cells.

In [ ]:
common_keys = []

for key, tool_map in rec.items():
    if "author" in tool_map and all(tool in tool_map for tool in MAJOR_TOOLS):
        common_keys.append(key)

common_summary = []

for tool in MAJOR_TOOLS:
    pairs = []
    for key in common_keys:
        tool_map = rec[key]
        author = tool_map["author"]
        pred = tool_map[tool]
        pairs.append((
            author["cell_id"],
            pred["cell_id"],
            as_float(pred["confidence_score"]),
            author["cell_label"],
            pred["cell_label"],
            pred["match_type"],
            author["organ"],
            author["dataset"],
            author["cell"],
        ))

    common_summary.append({"tool": tool, **metrics_for(pairs)})

print(f"Common author + all-major-tools cells: {len(common_keys):,}")
try:
    common_df = pd.DataFrame(common_summary)
    display(common_df)
except (NameError, ImportError):
    for row in common_summary:
        print(row)

## 10. Write all benchmark outputs

The notebook writes the same core result files used in the comparison, plus a common-cell summary.

In [ ]:
def write_csv(path, rows, fields=None):
    if fields is None:
        fields = list(rows[0].keys()) if rows else []

    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)


write_csv(OUTPUT_DIR / "ctann_vs_author_summary.csv", summary)
write_csv(OUTPUT_DIR / "ctann_vs_author_by_organ.csv", organ_summary)
write_csv(OUTPUT_DIR / "ctann_vs_author_by_dataset.csv", dataset_summary)
write_csv(OUTPUT_DIR / "ctann_vs_author_top_confusions.csv", confusion_rows)
write_csv(OUTPUT_DIR / "ctann_vs_author_confidence_bins.csv", confidence_summary)
write_csv(OUTPUT_DIR / "ctann_vs_author_common_cells.csv", common_summary)

with gzip.open(
    OUTPUT_DIR / "ctann_vs_author_mismatches.csv.gz",
    "wt",
    newline="",
    encoding="utf-8",
) as f:
    fields = list(mismatch_rows[0].keys()) if mismatch_rows else []
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    writer.writerows(mismatch_rows)

print("Wrote:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(" -", path)

## 11. Important interpretation note: ontology-aware evaluation

The strict benchmark above counts different CL IDs as a mismatch even if one term is a parent or child of the other. For example, a broad author annotation such as a proximal-tubule epithelial cell and a segment-specific Azimuth prediction may be biologically compatible.

A stronger next step is to use the Cell Ontology graph and classify each pair as:

```text
EXACT
NARROWER_THAN_AUTHOR
BROADER_THAN_AUTHOR
RELATED
UNRELATED
```

That ontology-aware evaluation should be analyzed separately from strict exact-ID agreement rather than silently changing the definition of accuracy.

## 12. Meaningful visualizations

The plots below are designed to answer specific benchmarking questions:

1. **Coverage vs exact agreement** — does a tool perform well only because it annotates a smaller subset of cells?
2. **Organ × tool heatmap** — where does each method perform well or poorly?
3. **Common-cell head-to-head** — which tool wins when all methods are evaluated on exactly the same cells?
4. **Confidence vs agreement** — is a tool's confidence score actually associated with agreement with the author annotation?
5. **Top author → prediction confusions** — which disagreements dominate each tool's error profile?

All figures use the strict Cell Ontology ID benchmark from the analysis above. Therefore, ontology parent/child relationships are still counted as disagreements.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# A consistent display order and readable names.
TOOL_ORDER = ["celltypist", "azimuth", "pan-human-azimuth", "popv"]
TOOL_LABELS = {
    "celltypist": "CellTypist",
    "azimuth": "Azimuth",
    "pan-human-azimuth": "Pan-Human Azimuth",
    "popv": "popV",
    "frmatch": "FR-Match",
}

FIGURE_DIR = OUTPUT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Notebook-friendly defaults. No explicit color palette is imposed.
plt.rcParams.update({
    "figure.figsize": (9, 5.5),
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

print("Figures will be written to:", FIGURE_DIR.resolve())


### 12.1 Coverage versus exact agreement

This plot prevents a common benchmarking mistake: comparing accuracy without considering how many author-annotated cells a tool actually covers. A strong method should ideally appear high on **both** measures.


In [ ]:
plot_df = (
    summary_df[summary_df["tool"].isin(TOOL_ORDER)]
    .set_index("tool")
    .reindex(TOOL_ORDER)
    .reset_index()
)

x = np.arange(len(plot_df))
width = 0.36

fig, ax = plt.subplots(figsize=(10, 5.5))
coverage_bars = ax.bar(
    x - width / 2,
    100 * plot_df["author_coverage"],
    width,
    label="Author-cell coverage",
)
agreement_bars = ax.bar(
    x + width / 2,
    100 * plot_df["exact_agreement"],
    width,
    label="Exact CL-ID agreement",
)

ax.set_ylabel("Percent of cells")
ax.set_title("Coverage and strict CT agreement are different questions")
ax.set_xticks(x, [TOOL_LABELS[t] for t in plot_df["tool"]])
ax.set_ylim(0, 100)
ax.legend()
ax.grid(axis="y", alpha=0.2)

for bars in (coverage_bars, agreement_bars):
    for bar in bars:
        value = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            value + 1.5,
            f"{value:.1f}%",
            ha="center",
            va="bottom",
            fontsize=9,
        )

fig.tight_layout()
fig.savefig(FIGURE_DIR / "01_coverage_vs_exact_agreement.png", dpi=180, bbox_inches="tight")
plt.show()


### 12.2 Organ × tool exact-agreement heatmap

This is the most important diagnostic visualization for this benchmark. It exposes tissue specificity that disappears in an overall average. Blank cells mean that no overlapping author/tool annotations were available for that organ.


In [ ]:
organ_matrix = (
    organ_df[organ_df["tool"].isin(TOOL_ORDER)]
    .pivot(index="organ", columns="tool", values="exact_agreement")
    .reindex(columns=TOOL_ORDER)
)

# Keep a biologically intuitive stable order when present.
organ_order = [o for o in ["Kidney", "Liver", "Lung", "Pancreas"] if o in organ_matrix.index]
organ_matrix = organ_matrix.reindex(organ_order)

values = 100 * organ_matrix.to_numpy(dtype=float)
masked = np.ma.masked_invalid(values)

fig, ax = plt.subplots(figsize=(9.5, 5.2))
im = ax.imshow(masked, aspect="auto", vmin=0, vmax=100)

ax.set_xticks(np.arange(len(organ_matrix.columns)))
ax.set_xticklabels([TOOL_LABELS[t] for t in organ_matrix.columns], rotation=25, ha="right")
ax.set_yticks(np.arange(len(organ_matrix.index)))
ax.set_yticklabels(organ_matrix.index)
ax.set_title("Exact CL-ID agreement varies strongly by organ")

for i in range(values.shape[0]):
    for j in range(values.shape[1]):
        val = values[i, j]
        if np.isfinite(val):
            # Let matplotlib use standard text rendering; add a compact cell label.
            ax.text(j, i, f"{val:.1f}%", ha="center", va="center", fontsize=10)
        else:
            ax.text(j, i, "—", ha="center", va="center", fontsize=10)

cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Exact CL-ID agreement (%)")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "02_organ_tool_heatmap.png", dpi=180, bbox_inches="tight")
plt.show()


### 12.3 Fair head-to-head comparison on common cells

Overall scores use different overlapping cell populations. This chart removes that coverage confound by restricting evaluation to the cells annotated by the author **and all four major CTann tools**.


In [ ]:
common_plot = (
    common_df[common_df["tool"].isin(TOOL_ORDER)]
    .set_index("tool")
    .reindex(TOOL_ORDER)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(9, 5.2))
bars = ax.bar(
    [TOOL_LABELS[t] for t in common_plot["tool"]],
    100 * common_plot["exact_agreement"],
)

ax.set_ylabel("Exact CL-ID agreement (%)")
ax.set_ylim(0, 100)
ax.set_title(f"Head-to-head agreement on the same {int(common_plot['overlap_cells'].iloc[0]):,} cells")
ax.grid(axis="y", alpha=0.2)

for bar, value in zip(bars, 100 * common_plot["exact_agreement"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        value + 1.5,
        f"{value:.1f}%",
        ha="center",
        va="bottom",
        fontsize=10,
    )

fig.tight_layout()
fig.savefig(FIGURE_DIR / "03_common_cell_head_to_head.png", dpi=180, bbox_inches="tight")
plt.show()


### 12.4 Confidence versus exact agreement

This plot checks whether higher model confidence corresponds to greater author agreement. The point labels give the number of cells in each confidence bin, which is important because a high rate based on only a few cells can be misleading.


In [ ]:
bin_order = [label for _, _, label in CONFIDENCE_BINS]
conf_plot = confidence_df[confidence_df["tool"].isin(TOOL_ORDER)].copy()
conf_plot["confidence_bin"] = pd.Categorical(
    conf_plot["confidence_bin"], categories=bin_order, ordered=True
)

fig, ax = plt.subplots(figsize=(11, 6))

for tool in TOOL_ORDER:
    d = conf_plot[conf_plot["tool"] == tool].sort_values("confidence_bin")
    if d.empty:
        continue
    x = np.arange(len(d))
    y = 100 * d["exact_agreement"].to_numpy()
    ax.plot(x, y, marker="o", label=TOOL_LABELS[tool])

    for xi, yi, n in zip(x, y, d["overlap_cells"]):
        ax.annotate(
            f"n={int(n):,}",
            (xi, yi),
            xytext=(0, 7),
            textcoords="offset points",
            ha="center",
            fontsize=7,
        )

ax.set_xticks(np.arange(len(bin_order)))
ax.set_xticklabels(bin_order, rotation=30, ha="right")
ax.set_xlabel("CTann confidence bin")
ax.set_ylabel("Exact CL-ID agreement (%)")
ax.set_ylim(0, 100)
ax.set_title("Does higher CTann confidence imply stronger author agreement?")
ax.legend()
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "04_confidence_vs_agreement.png", dpi=180, bbox_inches="tight")
plt.show()


### 12.5 Dominant author → CTann confusions

A single accuracy number does not explain *why* a tool disagrees. These ranked charts show the most frequent mismatch pairs. Large bars can reveal systematic granularity differences, synonym/ontology mapping issues, or biologically important confusion patterns.

A separate figure is drawn for each tool to keep the long CT labels readable.


In [ ]:
TOP_N = 10

for tool in TOOL_ORDER:
    d = (
        confusion_df[confusion_df["tool"] == tool]
        .sort_values(["count", "rank"], ascending=[False, True])
        .head(TOP_N)
        .copy()
    )
    if d.empty:
        continue

    # Shorten very long labels while preserving both sides of the confusion.
    def shorten(text, n=42):
        text = str(text)
        return text if len(text) <= n else text[: n - 1] + "…"

    d["pair"] = [
        f"{shorten(a)}  →  {shorten(p)}"
        for a, p in zip(d["author_cell_label"], d["pred_cell_label"])
    ]
    d = d.sort_values("count", ascending=True)

    fig, ax = plt.subplots(figsize=(12, 6.2))
    bars = ax.barh(d["pair"], d["count"])
    ax.set_xlabel("Number of mismatched cells")
    ax.set_title(f"Top strict-ID confusions — {TOOL_LABELS[tool]}")
    ax.grid(axis="x", alpha=0.2)

    for bar, count in zip(bars, d["count"]):
        ax.text(
            count,
            bar.get_y() + bar.get_height() / 2,
            f"  {int(count):,}",
            va="center",
            fontsize=9,
        )

    fig.tight_layout()
    safe_tool = tool.replace("-", "_")
    fig.savefig(
        FIGURE_DIR / f"05_top_confusions_{safe_tool}.png",
        dpi=180,
        bbox_inches="tight",
    )
    plt.show()


### 12.6 Suggested interpretation

Use the figures together rather than choosing a winner from one metric:

- **Coverage + exact agreement** describes how broadly and how strictly a tool matches author labels.
- The **organ heatmap** identifies where performance is tissue-dependent.
- The **common-cell plot** is the fairest direct tool comparison available from this file.
- The **confidence plot** tests whether confidence scores are useful for filtering predictions.
- The **confusion plots** identify systematic disagreements that should be reviewed with Cell Ontology relationships.

The next methodological upgrade should be an ontology-aware benchmark with categories such as `EXACT`, `NARROWER_THAN_AUTHOR`, `BROADER_THAN_AUTHOR`, `RELATED`, and `UNRELATED`. The same visualization framework can then show biologically compatible agreement separately from strict exact-ID agreement.
